In [11]:
# --------------------------
# Configuration & Path Setup
# --------------------------

import pandas as pd
import numpy as np
import os
from pathlib import Path
from typing import Union, Optional

# Cross-platform path resolution (consistent with Iteration 0)
def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    """
    Search upward from start_path for a directory containing 'Datasets' folder.
    This makes the notebook work on any system (macOS, Windows, Linux).
    """
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Datasets').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not find project root with 'Datasets' folder within {max_levels} levels. "
        f"Set THESIS_BASE_DIR environment variable or ensure Datasets folder exists."
    )

# Try environment variable first, then search for project root
env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base)
    print(f"Using THESIS_BASE_DIR from environment: {BASE_DIR}")
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())
    print(f"Found project root: {BASE_DIR}")

PATHS = {
    'raw_data': BASE_DIR / "Datasets" / "RW_Datasets",
    'iteration_output': BASE_DIR / "Datasets" / "Iteration_Outputs" / "_iteration_1",
    'embeddings': BASE_DIR / "Datasets" / "Embeddings" / "_iteration_1" / "bert-base-cased",
    'results': BASE_DIR / "Results" / "_iteration_1"
}

# Normalize all paths to absolute Path objects
for k, p in list(PATHS.items()):
    PATHS[k] = Path(p).resolve()

# Create output directories
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

# Data files configuration (JSONL files)
DATA_FILES = {
    'df_14': PATHS['raw_data'] / "14K_Reports_RW_ACTUALS_PS.jsonl",
    'df_28': PATHS['raw_data'] / "28k_Reports_RW_ACTUALS_ALL.jsonl",
    'df_30': PATHS['raw_data'] / "30K_Reports_RW_ACTUALS_ALL_v2.0.jsonl",
    'df_700': PATHS['raw_data'] / "700k_reports_RW_ACTUALS_NH_OBS.jsonl"
}

def load_data(file_path: Union[str, Path], low_memory: bool = False) -> Optional[pd.DataFrame]:
    """
    Load JSONL, CSV, or XLSX data into a DataFrame with error handling.
    
    Parameters: 
        file_path (Union[str, Path]): Path to the data file
        low_memory (bool): Pass to pd.read_csv (ignored for XLSX/JSONL)
        
    Returns:
        pandas.DataFrame: Loaded data or None if file not found
    """
    file_path = Path(file_path)  # Ensure it's a Path object
    
    # Check if file is JSONL
    if file_path.suffix.lower() == '.jsonl':
        try:
            df = pd.read_json(file_path, lines=True)
            print(f"Successfully loaded {file_path.name} ({len(df)} rows) [JSONL format]")
            return df
        except Exception as e:
            print(f"An error occurred while loading {file_path.name}: {e}")
            return None
    
    # Check if file is XLSX
    if file_path.suffix.lower() in ['.xlsx', '.xls']:
        try:
            df = pd.read_excel(file_path, engine='openpyxl')
            print(f"Successfully loaded {file_path.name} ({len(df)} rows) [Excel format]")
            return df
        except Exception as e:
            print(f"An error occurred while loading {file_path.name}: {e}")
            return None
    
    # For CSV files - try multiple encodings
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, low_memory=low_memory, encoding=encoding, on_bad_lines='skip')
            print(f"Successfully loaded {file_path.name} ({len(df)} rows) [encoding: {encoding}]")
            return df
        except (UnicodeDecodeError, Exception):
            if encoding == encodings[-1]:
                # Last attempt: try with error handling
                try:
                    df = pd.read_csv(file_path, low_memory=low_memory, encoding=encoding, 
                                    errors='ignore', on_bad_lines='skip')
                    print(f"Successfully loaded {file_path.name} ({len(df)} rows) [encoding: {encoding} with errors='ignore']")
                    return df
                except Exception as final_error:
                    print(f"An error occurred while loading {file_path.name}: {final_error}")
                    return None
            continue
    
    return None

# Load Each File (JSONL files)
RW_ACTUALS_PS = load_data(DATA_FILES['df_14'])
RW_ACTUALS_ALL_28 = load_data(DATA_FILES['df_28'])
RW_ACTUALS_ALL_30 = load_data(DATA_FILES['df_30'])
RW_ACTUALS_NH_OBS = load_data(DATA_FILES['df_700'])

# Check if all dataframes were loaded successfully
if all(df is not None for df in [RW_ACTUALS_PS, RW_ACTUALS_ALL_28, RW_ACTUALS_ALL_30, RW_ACTUALS_NH_OBS]):
    print("\n[OK] All files loaded successfully!")
else:
    print("\n[WARNING] Some files failed to load. Please check the file paths and try again.")

Found project root: /Users/shariarimrozekhan/Documents/GitHub/masterThesis2026
Successfully loaded 14K_Reports_RW_ACTUALS_PS.jsonl (14432 rows) [JSONL format]
Successfully loaded 28k_Reports_RW_ACTUALS_ALL.jsonl (28323 rows) [JSONL format]
Successfully loaded 30K_Reports_RW_ACTUALS_ALL_v2.0.jsonl (30609 rows) [JSONL format]
Successfully loaded 700k_reports_RW_ACTUALS_NH_OBS.jsonl (717041 rows) [JSONL format]

[OK] All files loaded successfully!


In [12]:
# =============================================================================
# Schema Alignment: Apply fixes from the column mapping comments
# =============================================================================

# --- 700K Dataset Fixes ---
# Rename DATETIME → CASE_OCCURENCE_DATE (comment: "Rename in 700k Reports")
rename_map_700k = {
    "DATETIME": "CASE_OCCURENCE_DATE",
}
RW_ACTUALS_NH_OBS = RW_ACTUALS_NH_OBS.rename(
    columns={k: v for k, v in rename_map_700k.items() if k in RW_ACTUALS_NH_OBS.columns}
)

# --- 30K Dataset Fixes ---
# 1) Rename columns (comment: "Rename in 30k Reports")
rename_map_30k = {
    "EMPLOYMENT_CATEGORY": "COMPANY_INVOLVED_TYPE",
    "HAZARD_PHYSICAL_SECURITY_EVENT": "HAZARD",
}
RW_ACTUALS_ALL_30 = RW_ACTUALS_ALL_30.rename(
    columns={k: v for k, v in rename_map_30k.items() if k in RW_ACTUALS_ALL_30.columns}
)

# 2) Add missing column (comment: "Column Missing. Added column in 30K Reports.")
if "IMM_ACTION_TAKEN_RECOM" not in RW_ACTUALS_ALL_30.columns:
    RW_ACTUALS_ALL_30["IMM_ACTION_TAKEN_RECOM"] = pd.NA

# 3) Delete columns marked "Delete from 30k"
drop_cols_30k = [
    "PERSONAL_INJURIES",
    "ACTUAL_WORKDAYS_LTA",
    "COMPANY_NAME",
    "COMPANY_TYPE",
    "CASE_APPROVED_DATE",
    "OUTSIDE_LEGAL_INFLUENCE",
    "RETURNED_TO_WORK_DATE",
]
RW_ACTUALS_ALL_30 = RW_ACTUALS_ALL_30.drop(
    columns=[c for c in drop_cols_30k if c in RW_ACTUALS_ALL_30.columns]
)

# Keep alias used in later cells
RW_ACTUALS_ALL = RW_ACTUALS_ALL_30

# --- Validation ---
print("=" * 60)
print("SCHEMA ALIGNMENT RESULTS")
print("=" * 60)

print("\n[700K] RW_ACTUALS_NH_OBS:")
print(f"  Shape: {RW_ACTUALS_NH_OBS.shape}")
print(f"  CASE_OCCURENCE_DATE: {'OK' if 'CASE_OCCURENCE_DATE' in RW_ACTUALS_NH_OBS.columns else 'MISSING'}")
print(f"  DATETIME: {'STILL PRESENT (unexpected)' if 'DATETIME' in RW_ACTUALS_NH_OBS.columns else 'REMOVED (renamed)'}")

print("\n[30K] RW_ACTUALS_ALL_30:")
print(f"  Shape: {RW_ACTUALS_ALL_30.shape}")
for c in ["COMPANY_INVOLVED_TYPE", "HAZARD", "IMM_ACTION_TAKEN_RECOM"]:
    print(f"  {c}: {'OK' if c in RW_ACTUALS_ALL_30.columns else 'MISSING'}")
print("  Columns removed:")
for c in drop_cols_30k:
    print(f"    {c}: {'REMOVED' if c not in RW_ACTUALS_ALL_30.columns else 'STILL PRESENT'}")

SCHEMA ALIGNMENT RESULTS

[700K] RW_ACTUALS_NH_OBS:
  Shape: (717041, 36)
  CASE_OCCURENCE_DATE: OK
  DATETIME: REMOVED (renamed)

[30K] RW_ACTUALS_ALL_30:
  Shape: (30609, 36)
  COMPANY_INVOLVED_TYPE: OK
  HAZARD: OK
  IMM_ACTION_TAKEN_RECOM: OK
  Columns removed:
    PERSONAL_INJURIES: REMOVED
    ACTUAL_WORKDAYS_LTA: REMOVED
    COMPANY_NAME: REMOVED
    COMPANY_TYPE: REMOVED
    CASE_APPROVED_DATE: REMOVED
    OUTSIDE_LEGAL_INFLUENCE: REMOVED
    RETURNED_TO_WORK_DATE: REMOVED


In [17]:
# =============================================================================
# Merge All Four Datasets
# =============================================================================

# Concatenate all four dataframes (14K, 28K, 30K, 700K)
master_df = pd.concat(
    [RW_ACTUALS_PS, RW_ACTUALS_ALL_28, RW_ACTUALS_ALL_30, RW_ACTUALS_NH_OBS],
    ignore_index=True
)

print(f"Combined master_df (before removing duplicates): {master_df.shape}")
print(f"Total records before deduplication: {len(master_df):,}")

# Per-source breakdown
print(f"\nPer-source record counts:")
print(f"  14K (PS):       {len(RW_ACTUALS_PS):>10,}")
print(f"  28K (ALL):      {len(RW_ACTUALS_ALL_28):>10,}")
print(f"  30K (ALL v2):   {len(RW_ACTUALS_ALL_30):>10,}")
print(f"  700K (NH_OBS):  {len(RW_ACTUALS_NH_OBS):>10,}")
print(f"  {'─'*35}")
print(f"  Total:          {len(master_df):>10,}")

print(f"\nTotal columns: {len(master_df.columns)}")
print(f"Columns: {list(master_df.columns)}")

Combined master_df (before removing duplicates): (790405, 36)
Total records before deduplication: 790,405

Per-source record counts:
  14K (PS):           14,432
  28K (ALL):          28,323
  30K (ALL v2):       30,609
  700K (NH_OBS):     717,041
  ───────────────────────────────────
  Total:             790,405

Total columns: 36
Columns: ['CASENO', 'COMPANY', 'FUNCTIONAL_GROUP', 'FUNCTION', 'FUNCTIONAL_AREA', 'FUNCTIONAL_LOCATION', 'FUNCTIONAL_SUB_LOCATION', 'LOCATION_SID', 'LOCATION_SHORT', 'COUNTRY_SHORT', 'SL_COUNTRY', 'SL_LOCATION_LVL_1', 'SL_LOCATION_LVL_2', 'SL_LOCATION_LVL_3', 'SL_LOCATION_LVL_4', 'CASE_OCCURENCE_DATE', 'TITLE', 'COMPANY_INVOLVED_TYPE', 'CASE_TYPE', 'CASE_SEVERITY', 'CASE_DESCRIPTION', 'STATUS', 'HAZARD', 'IMM_ACTION_TAKEN_RECOM', 'FULL_INVESTIGATION_DONE', 'CREATED_DATE', 'MODIFIED_DATE', 'APPROVED_WITHIN_DEADLINE', 'CASE_CLOSED_DATE', 'CASES_NO_OF_REGISTRATIONS', 'VALID_FROM', 'VALID_TO', 'POTENTIAL_SEV_LEVEL', 'RISK_AREA', 'LEARNINGS_ACTUAL_SEVERITY', 'LE

In [18]:
# Remove duplicates in CASENO column
duplicates_count = master_df['CASENO'].duplicated().sum()
print(f"Duplicates found in CASENO: {duplicates_count}")

# Keep the first occurrence of each CASENO
master_df = master_df.drop_duplicates(subset=['CASENO'], keep='last')

print(f"After removing duplicates: {len(master_df)} records")
print(f"Removed {duplicates_count} duplicate records")

Duplicates found in CASENO: 755479
After removing duplicates: 34926 records
Removed 755479 duplicate records


In [20]:
# =============================================================================
# Convert CSV to JSONL and load back as DataFrame
# =============================================================================

import pandas as pd

# Convert CSV → JSONL
df = pd.read_csv('/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/master_dataset.csv', low_memory=False)
df.to_json('/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/master_dataset.jsonl', orient='records', lines=True, force_ascii=False)
print(f'Converted {len(df)} rows to master_dataset.jsonl')

# Load the JSONL file into a DataFrame
master_df = pd.read_json('/Users/shariarimrozekhan/Documents/GitHub/masterThesis2026/master_dataset.jsonl', lines=True)
print(f'\nLoaded master_dataset.jsonl into DataFrame')
print(f'Shape: {master_df.shape}  ({master_df.shape[0]} rows, {master_df.shape[1]} columns)')

Converted 54648 rows to master_dataset.jsonl

Loaded master_dataset.jsonl into DataFrame
Shape: (54648, 36)  (54648 rows, 36 columns)


In [21]:
# =============================================================================
# Check Uniqueness in master_df
# =============================================================================

total_rows = len(master_df)
unique_caseno = master_df['CASENO'].nunique()
duplicate_caseno = master_df['CASENO'].duplicated().sum()

print(f"Total rows:        {total_rows:,}")
print(f"Unique CASENO:     {unique_caseno:,}")
print(f"Duplicate CASENO:  {duplicate_caseno:,}")
print(f"\nAll entries unique: {'YES' if duplicate_caseno == 0 else 'NO'}")

Total rows:        54,648
Unique CASENO:     31,778
Duplicate CASENO:  22,870

All entries unique: NO


In [22]:
# =============================================================================
# Inspect 5 Duplicate CASENO entries
# =============================================================================

# Find CASENOs that appear more than once
dup_casenos = master_df[master_df['CASENO'].duplicated(keep=False)]['CASENO'].unique()[:5]

print(f"Showing all rows for {len(dup_casenos)} duplicate CASENOs:\n")

for caseno in dup_casenos:
    rows = master_df[master_df['CASENO'] == caseno][['CASENO', 'TITLE', 'CASE_DESCRIPTION']]
    print(f"{'='*80}")
    print(f"CASENO: {caseno}  ({len(rows)} occurrences)")
    print(f"{'='*80}")
    for i, (_, row) in enumerate(rows.iterrows(), 1):
        print(f"  [{i}] TITLE: {str(row['TITLE'])[:100]}")
        print(f"      DESC:  {str(row['CASE_DESCRIPTION'])[:150]}")
    print()

Showing all rows for 5 duplicate CASENOs:

CASENO: 5709  (2 occurrences)
  [1] TITLE: IT-Probleme: keine Netzanbindung von mindestens 6 Rechnern
      DESC:  seit Freitag (ca. 12:00 Uhr) besteht für mindestens 6 Rechner keine Netzanbindung mehr (seit 5 Tagen).
Dadurch ist z.B.keine Bearbeitung von Rechnung
  [2] TITLE: IT-Probleme: keine Netzanbindung von mindestens 6 Rechnern
      DESC:  seit Freitag (ca. 12:00 Uhr) besteht für mindestens 6 Rechner keine Netzanbindung mehr (seit 5 Tagen).
Dadurch ist z.B.keine Bearbeitung von Rechnung

CASENO: 5187  (2 occurrences)
  [1] TITLE: Vorfall UST Dispatching Ausfall der USV
      DESC:  Am Donnerstagmorgen, den 25.07.2019 um 02:00 Uhr fiel die USV des Dispatchings in Düsseldorf im Float Gebäude A1 im 3. Stockwerk aus, d.h. kein Netzwe
  [2] TITLE: Vorfall UST Dispatching Ausfall der USV
      DESC:  Am Donnerstagmorgen, den 25.07.2019 um 02:00 Uhr fiel die USV des Dispatchings in Düsseldorf im Float Gebäude A1 im 3. Stockwerk aus, d.h. kein

In [15]:
# =============================================================================
# Data Cleaning
# =============================================================================

print(f"Before cleaning: {len(master_df)} records")

# Remove duplicates based on CASENO (keep last occurrence)
if 'CASENO' in master_df.columns:
    master_df = master_df.drop_duplicates(subset=['CASENO'], keep='last')
    print(f"After removing duplicates: {len(master_df)} records")

# Remove rows with missing critical text fields
required_cols = ['TITLE', 'CASE_DESCRIPTION', 'CASE_TYPE']
for col in required_cols:
    if col in master_df.columns:
        before = len(master_df)
        master_df = master_df[master_df[col].notna() & (master_df[col].astype(str).str.strip() != '')]
        print(f"After removing missing {col}: {len(master_df)} records (removed {before - len(master_df)})")

# Remove completely empty rows
master_df = master_df.dropna(how='all')
print(f"\nAfter all cleaning: {len(master_df)} records")

Before cleaning: 34704 records
After removing duplicates: 34704 records
After removing missing TITLE: 34704 records (removed 0)
After removing missing CASE_DESCRIPTION: 34704 records (removed 0)
After removing missing CASE_TYPE: 34396 records (removed 308)

After all cleaning: 34396 records


In [16]:
# =============================================================================
# EDA: Number of Records per Original Dataset
# =============================================================================

import plotly.graph_objects as go
import plotly.express as px

dataset_counts = {
    '14k PS Reports': df_ps.shape[0],
    '28k ALL Reports': df_all.shape[0],
    '900k NH_OBS Reports': df_nh_obs.shape[0]
}

fig = go.Figure(data=[go.Bar(
    x=list(dataset_counts.keys()),
    y=list(dataset_counts.values()),
    text=list(dataset_counts.values()),
    textposition='auto',
    marker=dict(color=['#08519c', '#3182bd', '#6baed6'])
)])

fig.update_layout(
    title='Number of Records per Source Dataset',
    xaxis_title='Dataset',
    yaxis_title='Number of Records',
    height=400,
    showlegend=False,
    hovermode='x unified'
)

fig.show()


NameError: name 'df_ps' is not defined

In [ ]:
# =============================================================================
# EDA: Data Quality Overview (Missing Values)
# =============================================================================

import plotly.express as px

# Calculate missing values percentage for key columns
missing_data = {
    col: (master_df[col].isna().sum() / len(master_df) * 100) 
    for col in master_df.columns 
    if master_df[col].isna().sum() > 0
}

# Sort by missing percentage descending
missing_data = dict(sorted(missing_data.items(), key=lambda x: x[1], reverse=True)[:15])

if missing_data:
    fig = go.Figure(data=[go.Bar(
        y=list(missing_data.keys()),
        x=list(missing_data.values()),
        orientation='h',
        marker=dict(color=list(missing_data.values()),
                   colorscale='Blues')
    )])
    
    fig.update_layout(
        title='Missing Values by Column (Top 15)',
        xaxis_title='Missing Value %',
        yaxis_title='Column',
        height=500,
        showlegend=False,
        hovermode='y unified'
    )
    fig.show()
else:
    print("No missing values in the dataset")


In [ ]:
# =============================================================================
# EDA: Record Count Before and After Cleaning
# =============================================================================

initial_count = {
    'df_ps': df_ps.shape[0],
    'df_all': df_all.shape[0],
    'df_nh_obs': df_nh_obs.shape[0],
}

total_initial = sum(initial_count.values())
after_merge = master_df.shape[0] if 'master_df' in dir() else len(master_df)

cleaning_stages = {
    'Initial (Combined)': total_initial,
    'After Dedup & Merge': len(master_df) if 'master_df' in dir() else after_merge,
}

fig = go.Figure(data=[go.Bar(
    x=list(cleaning_stages.keys()),
    y=list(cleaning_stages.values()),
    text=[f'{v:,}' for v in cleaning_stages.values()],
    textposition='auto',
    marker=dict(color=['#c6dbef', '#08519c'])
)])

fig.update_layout(
    title='Data Pipeline: Records Through Processing Stages',
    xaxis_title='Processing Stage',
    yaxis_title='Number of Records',
    height=400,
    showlegend=False,
    hovermode='x unified'
)

fig.show()


In [ ]:
# =============================================================================
# EDA: Case Type Distribution
# =============================================================================

if 'CASE_TYPE' in master_df.columns:
    case_counts = master_df['CASE_TYPE'].value_counts()
    
    fig = go.Figure(data=[go.Bar(
        y=case_counts.index.astype(str),
        x=case_counts.values,
        orientation='h',
        marker=dict(
            color=case_counts.values,
            colorscale='Blues'
        ),
        text=case_counts.values,
        textposition='outside'
    )])
    
    fig.update_layout(
        title='Distribution of Case Types',
        xaxis_title='Number of Cases',
        yaxis_title='Case Type',
        height=500,
        showlegend=False,
        hovermode='y unified'
    )
    fig.show()
    
    print(f"\nTotal unique case types: {master_df['CASE_TYPE'].nunique()}")
    print(f"Top 5 case types:\n{case_counts.head()}")


In [ ]:
# =============================================================================
# EDA: Text Length Analysis
# =============================================================================

# Calculate text lengths before language detection
if 'TITLE' in master_df.columns and 'CASE_DESCRIPTION' in master_df.columns:
    master_df['title_length'] = master_df['TITLE'].astype(str).str.len()
    master_df['description_length'] = master_df['CASE_DESCRIPTION'].astype(str).str.len()
    master_df['combined_text_length'] = master_df['title_length'] + master_df['description_length']
    
    fig = go.Figure()
    
    fig.add_trace(go.Box(
        y=master_df['title_length'],
        name='Title Length',
        marker_color='#08519c'
    ))
    
    fig.add_trace(go.Box(
        y=master_df['description_length'],
        name='Description Length',
        marker_color='#3182bd'
    ))
    
    fig.add_trace(go.Box(
        y=master_df['combined_text_length'],
        name='Combined Text Length',
        marker_color='#6baed6'
    ))
    
    fig.update_layout(
        title='Text Length Distribution',
        yaxis_title='Character Count',
        height=500,
        hovermode='y unified',
        boxmode='group'
    )
    fig.show()
    
    print(f"Title length - Min: {master_df['title_length'].min()}, Max: {master_df['title_length'].max()}, Mean: {master_df['title_length'].mean():.0f}, Median: {master_df['title_length'].median():.0f}")
    print(f"Description length - Min: {master_df['description_length'].min()}, Max: {master_df['description_length'].max()}, Mean: {master_df['description_length'].mean():.0f}, Median: {master_df['description_length'].median():.0f}")
    print(f"Combined text length - Min: {master_df['combined_text_length'].min()}, Max: {master_df['combined_text_length'].max()}, Mean: {master_df['combined_text_length'].mean():.0f}, Median: {master_df['combined_text_length'].median():.0f}")

In [ ]:
# =============================================================================
# Language Detection (Filter to English only)
# =============================================================================

# Install langdetect if needed: pip install langdetect
from langdetect import detect, LangDetectException
from tqdm import tqdm

def detect_language(text):
    """Detect language of text. Returns language code or None."""
    try:
        if pd.isna(text) or str(text).strip() == '':
            return None
        return detect(str(text))
    except LangDetectException:
        return None

# Apply language detection (this may take a few minutes)
print("Detecting language for TITLE...")
tqdm.pandas(desc="TITLE")
master_df['title_lang'] = master_df['TITLE'].progress_apply(detect_language)

print("\nDetecting language for CASE_DESCRIPTION...")
tqdm.pandas(desc="CASE_DESCRIPTION")
master_df['description_lang'] = master_df['CASE_DESCRIPTION'].progress_apply(detect_language)

# Show language distribution
print("\nTitle language distribution:")
print(master_df['title_lang'].value_counts().head(10))
print("\nDescription language distribution:")
print(master_df['description_lang'].value_counts().head(10))

In [ ]:
# =============================================================================
# Filter to English-only Dataset
# =============================================================================

# Keep only records where BOTH title and description are English
english_df = master_df[
    (master_df['title_lang'] == 'en') & 
    (master_df['description_lang'] == 'en')
].copy()

print(f"Total records: {len(master_df)}")
print(f"English records: {len(english_df)} ({100*len(english_df)/len(master_df):.2f}%)")
print(f"Non-English excluded: {len(master_df) - len(english_df)} ({100*(len(master_df)-len(english_df))/len(master_df):.2f}%)")

# Create minimal dataset with key fields
english_dataset_minimal = english_df[['CASENO', 'TITLE', 'CASE_DESCRIPTION', 'CASE_TYPE']].copy()

# Show category distribution
print("\nCategory distribution in English dataset:")
print(english_dataset_minimal['CASE_TYPE'].value_counts())

In [ ]:
# =============================================================================
# EDA: Language Detection Distribution (Expected Languages Only)
# =============================================================================

# Define expected languages
expected_langs = {'en', 'de', 'nl', 'sv', 'hu', 'ru'}

# Filter to only expected languages
title_lang_filtered = master_df[master_df['title_lang'].isin(expected_langs)]['title_lang'].value_counts().sort_values(ascending=False)
desc_lang_filtered = master_df[master_df['description_lang'].isin(expected_langs)]['description_lang'].value_counts().sort_values(ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=title_lang_filtered.index,
    y=title_lang_filtered.values,
    name='Title Language',
    marker_color='#08519c'
))

fig.add_trace(go.Bar(
    x=desc_lang_filtered.index,
    y=desc_lang_filtered.values,
    name='Description Language',
    marker_color='#6baed6'
))

fig.update_layout(
    title='Language Distribution in Dataset (Expected Languages Only)',
    xaxis_title='Language Code',
    yaxis_title='Number of Records',
    height=500,
    barmode='group',
    hovermode='x unified'
)

fig.show()

print(f"\nExpected languages found in dataset:")
print(f"  Title languages: {title_lang_filtered.to_dict()}")
print(f"  Description languages: {desc_lang_filtered.to_dict()}")

# Summary
print(f"\nLanguage Coverage:")
print(f"  Records with expected language in title: {title_lang_filtered.sum():,}")
print(f"  Records with expected language in description: {desc_lang_filtered.sum():,}")
print(f"  Total records: {len(master_df):,}")

In [ ]:
# =============================================================================
# EDA: Language Code to Language Name Mapping (Expected Languages Only)
# =============================================================================

# ISO 639-1 language code mapping
language_names = {
    'en': 'English',
    'de': 'German',
    'fr': 'French',
    'es': 'Spanish',
    'it': 'Italian',
    'pt': 'Portuguese',
    'nl': 'Dutch',
    'sv': 'Swedish',
    'no': 'Norwegian',
    'da': 'Danish',
    'fi': 'Finnish',
    'el': 'Greek',
    'ru': 'Russian',
    'ar': 'Arabic',
    'ja': 'Japanese',
    'zh-cn': 'Chinese (Simplified)',
    'zh': 'Chinese',
    'ko': 'Korean',
    'tr': 'Turkish',
    'pl': 'Polish',
    'hu': 'Hungarian',
    'th': 'Thai',
    'vi': 'Vietnamese',
    'id': 'Indonesian',
    'uk': 'Ukrainian',
    'ro': 'Romanian',
    'cs': 'Czech',
    'sk': 'Slovak',
    'bg': 'Bulgarian',
    'hr': 'Croatian',
    'sr': 'Serbian',
    'sl': 'Slovenian',
}

# Define expected languages (6 languages from 8 datasets)
# en: UK, International, UAE
# de: Germany
# nl: Netherlands
# sv: Sweden
# hu: Hungary
# ru: Russia
expected_langs = {'en', 'de', 'nl', 'sv', 'hu', 'ru'}

# Create mapping table for expected languages only
lang_data = []
for code in sorted(expected_langs):
    lang_name = language_names.get(code, 'Unknown')
    title_count = (master_df['title_lang'] == code).sum()
    desc_count = (master_df['description_lang'] == code).sum()
    lang_data.append({
        'Language Code': code,
        'Language Name': lang_name,
        'Title Count': title_count,
        'Description Count': desc_count
    })

lang_df = pd.DataFrame(lang_data).sort_values('Title Count', ascending=False)

print("\n" + "=" * 80)
print("LANGUAGE DETECTION RESULTS - EXPECTED LANGUAGES ONLY")
print("=" * 80)
print(lang_df.to_string(index=False))
print("=" * 80)

In [ ]:
# =============================================================================
# DIAGNOSTIC: All Detected Languages vs Expected Languages
# =============================================================================

print("\n" + "=" * 80)
print("LANGUAGE DETECTION DIAGNOSTIC")
print("=" * 80)

# Get all detected languages
all_detected_langs = set(master_df['title_lang'].dropna().unique()) | set(master_df['description_lang'].dropna().unique())
print(f"\nTotal detected unique languages: {len(all_detected_langs)}")
print(f"All detected language codes: {sorted(all_detected_langs)}")

# Define the 8 expected languages
expected_langs = {'en', 'de', 'nl', 'sv', 'hu', 'ru'}  # 6 languages from the 8 datasets
print(f"\nExpected language codes: {sorted(expected_langs)}")

# Find languages detected but not expected
unexpected_langs = all_detected_langs - expected_langs
print(f"\nUnexpected languages detected ({len(unexpected_langs)}): {sorted(unexpected_langs)}")

# Count records for each
print(f"\n{'Language':<12} {'Title Count':>15} {'Desc Count':>15} {'Status':<15}")
print("-" * 60)

for code in sorted(all_detected_langs):
    title_count = (master_df['title_lang'] == code).sum()
    desc_count = (master_df['description_lang'] == code).sum()
    status = "EXPECTED" if code in expected_langs else "UNEXPECTED"
    print(f"{code:<12} {title_count:>15} {desc_count:>15} {status:<15}")

print("=" * 80)


In [ ]:
# =============================================================================
# EDA: Filtering Pipeline - Records at Each Stage
# =============================================================================

import plotly.graph_objects as go

# Calculate records at each filtering stage
total_master = len(master_df)
en_records = len(english_df) if 'english_df' in dir() else 0
en_minimal = len(english_dataset_minimal) if 'english_dataset_minimal' in dir() else 0

stages = {
    'Total\nCombined': total_master,
    'After Dedup\n& Cleaning': total_master,
    'English\nOnly': en_records,
    'Final Dataset\n(Minimal)': en_minimal
}

# Create waterfall effect visualization
fig = go.Figure(data=[go.Bar(
    x=list(stages.keys()),
    y=list(stages.values()),
    text=[f'{v:,}' for v in stages.values()],
    textposition='auto',
    marker=dict(color=['#08306b', '#08519c', '#3182bd', '#6baed6'])
)])

fig.update_layout(
    title='Data Filtering Pipeline - Records at Each Stage',
    yaxis_title='Number of Records',
    height=400,
    showlegend=False,
    hovermode='x unified'
)

fig.show()

print(f"Filtering efficiency: {en_records}/{total_master} ({100*en_records/total_master:.2f}%) records retained as English")


In [ ]:
# =============================================================================
# EDA: Binary Label Distribution (Process Safety vs Non-Process Safety)
# =============================================================================

if 'english_dataset_minimal' in dir() and 'binary_label' in english_dataset_minimal.columns:
    label_counts = english_dataset_minimal['binary_label'].value_counts().sort_index()
    labels = ['Non-Process Safety', 'Process Safety']
    colors = ['#08519c', '#3182bd']
    
    # Pie chart
    fig = go.Figure(data=[go.Pie(
        labels=labels,
        values=label_counts.values,
        marker=dict(colors=colors),
        textinfo='label+percent+value',
        textposition='auto',
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    )])
    
    fig.update_layout(
        title='Binary Label Distribution in English Dataset',
        height=500
    )
    
    fig.show()
    
    print(f"\nLabel Distribution:")
    print(f"  Non-Process Safety (0): {label_counts[0]:,} ({100*label_counts[0]/len(english_dataset_minimal):.2f}%)")
    print(f"  Process Safety (1): {label_counts[1]:,} ({100*label_counts[1]/len(english_dataset_minimal):.2f}%)")
    print(f"  Class Imbalance Ratio: {label_counts[0]/label_counts[1]:.2f}:1")


In [ ]:
# =============================================================================
# Step 5: Binary Classification Formulation
# =============================================================================

# Create binary label: 1 = Process Safety, 0 = Non-Process Safety
english_dataset_minimal['binary_label'] = (
    english_dataset_minimal['CASE_TYPE'] == 'Process Safety'
).astype(int)

# Create combined text feature
english_dataset_minimal['text_features'] = (
    english_dataset_minimal['TITLE'].astype(str) + '. ' + 
    english_dataset_minimal['CASE_DESCRIPTION'].astype(str)
)

print("Binary label distribution:")
print(english_dataset_minimal['binary_label'].value_counts())
print(f"\nPositive class (Process Safety): {english_dataset_minimal['binary_label'].sum()}")
print(f"Negative class (Non-Process Safety): {(english_dataset_minimal['binary_label'] == 0).sum()}")

# Save the prepared dataset
english_dataset_minimal.to_csv(OUTPUT_DIR / 'english_dataset_minimal.csv', index=False)
print(f"\nSaved to: {OUTPUT_DIR / 'english_dataset_minimal.csv'}")

In [ ]:
# =============================================================================
# BERT Embedding Generation
# =============================================================================

# Install if needed: pip install transformers torch
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm

# Initialize BERT-base-uncased
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Set to evaluation mode

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Using device: {device}")

def get_bert_embeddings(texts, batch_size=8, max_length=512):
    """Generate BERT [CLS] embeddings for a list of texts."""
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i + batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)
        
        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)
            # Extract [CLS] token embedding (first token)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_embeddings)
    
    return np.array(embeddings)

# Generate embeddings
texts = english_dataset_minimal['text_features'].tolist()
X = get_bert_embeddings(texts)
y = english_dataset_minimal['binary_label'].values

print(f"\nEmbeddings shape: {X.shape}")  # Expected: (n_samples, 768)
print(f"Labels shape: {y.shape}")

# Save embeddings for reuse
np.save(OUTPUT_DIR / 'bert_embeddings.npy', X)
np.save(OUTPUT_DIR / 'labels.npy', y)
print(f"Saved embeddings to: {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# Train/Test Split
# =============================================================================

from sklearn.model_selection import train_test_split

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"  - Non-Process Safety: {(y_train == 0).sum()} ({100*(y_train == 0).mean():.2f}%)")
print(f"  - Process Safety: {(y_train == 1).sum()} ({100*(y_train == 1).mean():.2f}%)")

print(f"\nTest set: {X_test.shape[0]} samples")
print(f"  - Non-Process Safety: {(y_test == 0).sum()} ({100*(y_test == 0).mean():.2f}%)")
print(f"  - Process Safety: {(y_test == 1).sum()} ({100*(y_test == 1).mean():.2f}%)")

In [ ]:
# =============================================================================
# Train Linear SVM Classifier
# =============================================================================

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Scale features (good practice for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Linear SVM tuned for ~0.76 Macro F1
svm_model = SVC(
    kernel='linear',
    C=0.0099,  # Very low regularization
    class_weight='balanced',  # Full class balancing
    random_state=42
)

print("Training SVM classifier...")
svm_model.fit(X_train_scaled, y_train)
print("Training complete!")

# Save model
import joblib
joblib.dump(svm_model, OUTPUT_DIR / 'svm_baseline_model.joblib')
joblib.dump(scaler, OUTPUT_DIR / 'scaler.joblib')
print(f"Model saved to: {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# EDA: Train/Test Split Distribution
# =============================================================================

if 'y_train' in dir() and 'y_test' in dir():
    train_dist = np.bincount(y_train)
    test_dist = np.bincount(y_test)
    
    labels = ['Non-Process Safety', 'Process Safety']
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=labels,
        y=train_dist,
        name='Training Set',
        marker_color='#08519c'
    ))
    
    fig.add_trace(go.Bar(
        x=labels,
        y=test_dist,
        name='Test Set',
        marker_color='#6baed6'
    ))
    
    fig.update_layout(
        title='Train/Test Split Distribution by Label',
        xaxis_title='Label',
        yaxis_title='Number of Samples',
        height=400,
        barmode='group',
        hovermode='x unified'
    )
    
    fig.show()
    
    print(f"Training Set Distribution:")
    print(f"  Non-Process Safety: {train_dist[0]:,} ({100*train_dist[0]/len(y_train):.2f}%)")
    print(f"  Process Safety: {train_dist[1]:,} ({100*train_dist[1]/len(y_train):.2f}%)")
    
    print(f"\nTest Set Distribution:")
    print(f"  Non-Process Safety: {test_dist[0]:,} ({100*test_dist[0]/len(y_test):.2f}%)")
    print(f"  Process Safety: {test_dist[1]:,} ({100*test_dist[1]/len(y_test):.2f}%)")


In [ ]:
# =============================================================================
# Model Evaluation
# =============================================================================

from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, f1_score
)

# Predictions
y_pred = svm_model.predict(X_test_scaled)

# Classification report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(
    y_test, y_pred, 
    target_names=['Non-Process Safety', 'Process Safety'],
    digits=4
)
print(report)

# Key metrics
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\nOverall Accuracy: {accuracy:.4f} ({100*accuracy:.2f}%)")
print(f"Macro F1-Score: {macro_f1:.4f}")
print(f"Weighted F1-Score: {weighted_f1:.4f}")

# Save report
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
    f.write(f"\nAccuracy: {accuracy:.4f}\n")
    f.write(f"Macro F1: {macro_f1:.4f}\n")
    f.write(f"Weighted F1: {weighted_f1:.4f}\n")

In [ ]:
# =============================================================================
# Confusion Matrix Analysis
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)
print(f"\n{'':20} Predicted")
print(f"{'':20} Non-PS    PS")
print(f"Actual Non-PS      {tn:5d}   {fp:5d}")
print(f"Actual PS          {fn:5d}   {tp:5d}")

print(f"\nTrue Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

# Error rates
fpr = fp / (fp + tn)  # False Positive Rate
fnr = fn / (fn + tp)  # False Negative Rate
print(f"\nFalse Positive Rate: {100*fpr:.2f}%")
print(f"False Negative Rate: {100*fnr:.2f}%")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Non-Process Safety', 'Process Safety'],
    yticklabels=['Non-Process Safety', 'Process Safety']
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Iteration 0: Confusion Matrix (BERT + SVM)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# =============================================================================
# Summary and Gap Analysis
# =============================================================================

import json

# RQ1 Target
RQ1_TARGET = 0.81

# Summary
summary = {
    'iteration': 0,
    'model': 'BERT-base-uncased + Linear SVM',
    'dataset': {
        'total_english_samples': len(english_dataset_minimal),
        'train_samples': len(y_train),
        'test_samples': len(y_test),
        'positive_class_ratio': float(y.mean())
    },
    'results': {
        'accuracy': float(accuracy),
        'macro_f1': float(macro_f1),
        'weighted_f1': float(weighted_f1),
        'precision_ps': float(tp / (tp + fp)),
        'recall_ps': float(tp / (tp + fn)),
        'precision_non_ps': float(tn / (tn + fn)),
        'recall_non_ps': float(tn / (tn + fp))
    },
    'confusion_matrix': {
        'tn': int(tn), 'fp': int(fp), 
        'fn': int(fn), 'tp': int(tp)
    },
    'gap_analysis': {
        'current_macro_f1': float(macro_f1),
        'rq1_target': RQ1_TARGET,
        'gap': float(RQ1_TARGET - macro_f1)
    }
}

# Save summary
with open(OUTPUT_DIR / 'iteration_0_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Print summary
print("=" * 60)
print("ITERATION 0 SUMMARY")
print("=" * 60)
print(f"\nModel: {summary['model']}")
print(f"Dataset: {summary['dataset']['total_english_samples']} English samples")
print(f"\nResults:")
print(f"  Accuracy: {summary['results']['accuracy']:.4f}")
print(f"  Macro F1: {summary['results']['macro_f1']:.4f}")
print(f"  Weighted F1: {summary['results']['weighted_f1']:.4f}")
print(f"\nGap Analysis (RQ1):")
print(f"  Current Macro F1: {macro_f1:.4f}")
print(f"  RQ1 Target: >= {RQ1_TARGET}")
print(f"  Gap to close: {RQ1_TARGET - macro_f1:+.4f}")

if macro_f1 >= RQ1_TARGET:
    print("\n✓ RQ1 TARGET ACHIEVED!")
else:
    print(f"\n→ Need to improve Macro F1 by {100*(RQ1_TARGET - macro_f1):.2f} percentage points")

print(f"\nResults saved to: {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# Diagnostic: Investigate duplicate removal
# =============================================================================

# 1) Check shape of each source dataframe BEFORE merge
print("=" * 60)
print("SOURCE DATAFRAME SHAPES")
print("=" * 60)
print(f"df_ps: {df_ps.shape[0]} rows")
print(f"df_all: {df_all.shape[0]} rows")
print(f"df_nh_obs: {df_nh_obs.shape[0]} rows")
print(f"Total if concatenated: {df_ps.shape[0] + df_all.shape[0] + df_nh_obs.shape[0]} rows")

# 2) Check unique CASENO in each source
print("\n" + "=" * 60)
print("UNIQUE CASENO IN EACH SOURCE")
print("=" * 60)
print(f"df_ps unique CASENO: {df_ps['CASENO'].nunique()}")
print(f"df_all unique CASENO: {df_all['CASENO'].nunique()}")
print(f"df_nh_obs unique CASENO: {df_nh_obs['CASENO'].nunique()}")

# 3) Check overlap between datasets (same CASENO appearing in multiple sources)
print("\n" + "=" * 60)
print("OVERLAP BETWEEN DATASETS")
print("=" * 60)
caseno_ps = set(df_ps['CASENO'].dropna().astype(str))
caseno_all = set(df_all['CASENO'].dropna().astype(str))
caseno_nh = set(df_nh_obs['CASENO'].dropna().astype(str))

print(f"Overlap df_ps & df_all: {len(caseno_ps & caseno_all)}")
print(f"Overlap df_ps & df_nh_obs: {len(caseno_ps & caseno_nh)}")
print(f"Overlap df_all & df_nh_obs: {len(caseno_all & caseno_nh)}")
print(f"In all three: {len(caseno_ps & caseno_all & caseno_nh)}")

# 4) Total unique CASENO across all sources
all_caseno = caseno_ps | caseno_all | caseno_nh
print(f"\nTotal UNIQUE CASENO across all sources: {len(all_caseno)}")

# 5) Check for duplicates WITHIN each source
print("\n" + "=" * 60)
print("DUPLICATES WITHIN EACH SOURCE")
print("=" * 60)
print(f"df_ps duplicates: {df_ps['CASENO'].duplicated().sum()}")
print(f"df_all duplicates: {df_all['CASENO'].duplicated().sum()}")
print(f"df_nh_obs duplicates: {df_nh_obs['CASENO'].duplicated().sum()}")

# 6) Check common_cols - are we losing data due to column filtering?
print("\n" + "=" * 60)
print("COLUMN INTERSECTION CHECK")
print("=" * 60)
print(f"df_ps columns: {len(df_ps.columns)}")
print(f"df_all columns: {len(df_all.columns)}")
print(f"df_nh_obs columns: {len(df_nh_obs.columns)}")
print(f"Common columns: {len(common_cols)}")
print(f"\nColumns in df_ps but NOT in common: {set(df_ps.columns) - common_cols}")
print(f"Columns in df_all but NOT in common: {set(df_all.columns) - common_cols}")
print(f"Columns in df_nh_obs but NOT in common: {set(df_nh_obs.columns) - common_cols}")

# 7) Verify CASENO is in common_cols
print(f"\n'CASENO' in common_cols: {'CASENO' in common_cols}")